In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

In [2]:
# Select only outbound tweets
outbound = df[df["inbound"] == False]

# Connect each outbound tweet to the tweet it is responding to
outbound_customer = outbound.merge(
    df[["tweet_id", "author_id"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_brand", "_customer")
)

# Count distinct customers for each brand
customer_counts = (
    outbound_customer
    .groupby("author_id_brand")["author_id_customer"]
    .nunique()
    .reset_index(name="distinct_customers")
)

# Sort by number of customers
customer_counts = customer_counts.sort_values(
    "distinct_customers",
    ascending=False
)

# Display top authors
customer_counts.head(20)

,author_id_brand,distinct_customers
10,AppleSupport,76366
8,AmazonHelp,71049
85,Uber_Support,38300
77,SpotifyCares,27794
40,Delta,22331
99,comcastcares,21824
9,AmericanAir,21686
78,TMobileHelp,19943
76,SouthwestAir,19713
26,Ask_Spectrum,17214


In [3]:
# Convert parent tweet ID to numeric

df["in_response_to_tweet_id"] = pd.to_numeric(
    df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

# Create mapping:
#    tweet_id → tweet id is replying to

parent = dict(
    zip(
        df["tweet_id"],
        df["in_response_to_tweet_id"]
    )
)

# Function to find the original/root tweet

def find_root(tweet_id):
    visited = set()

    while pd.notna(parent.get(tweet_id)):

        # Prevent infinite loops
        if tweet_id in visited:
            break

        visited.add(tweet_id)

        tweet_id = int(parent[tweet_id])

    return tweet_id

# Assign conversation ID to every tweet

df["conversation_id"] = df["tweet_id"].map(find_root)

# Define the 5 brands

brands = [
    "AppleSupport",
    "AmazonHelp",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

# Calculate conversation statistics

results = []

for brand in brands:

    # Tweets posted by this brand
    brand_tweets = df[
        df["author_id"] == brand
    ]

    # Conversation IDs involving this brand
    conversation_ids = brand_tweets[
        "conversation_id"
    ].unique()

    # Get all tweets belonging to those conversations
    conversations = df[
        df["conversation_id"].isin(conversation_ids)
    ]

    # Number of complete conversation threads
    number_of_conversations = (
        conversations["conversation_id"].nunique()
    )

    # Number of customer messages
    number_of_customer_messages = (
        conversations["inbound"] == True
    ).sum()

    # Number of brand responses
    number_of_brand_responses = (
        (conversations["author_id"] == brand) &
        (conversations["inbound"] == False)
    ).sum()

    # Store results
    results.append({
        "Brand": brand,
        "Number of conversations": number_of_conversations,
        "Number of customer messages": number_of_customer_messages,
        "Number of brand responses": number_of_brand_responses
    })


conversation_summary = pd.DataFrame(results)

print(conversation_summary)

          Brand  Number of conversations  Number of customer messages  \
0  AppleSupport                    80702                       131764   
1    AmazonHelp                    82534                       203598   
2  Uber_Support                    41923                        72154   
3  SpotifyCares                    28280                        48543   
4         Delta                    26166                        45296   

   Number of brand responses  
0                     106860  
1                     169840  
2                      56270  
3                      43265  
4                      42253  


In [4]:
conversation_summary["Average messages per conversation"] = (
    conversation_summary["Number of customer messages"]
    + conversation_summary["Number of brand responses"]
) / conversation_summary["Number of conversations"]

conversation_summary

,Brand,Number of conversations,Number of customer messages,Number of brand responses,Average messages per conversation
0,AppleSupport,80702,131764,106860,2.956854
1,AmazonHelp,82534,203598,169840,4.524657
2,Uber_Support,41923,72154,56270,3.063330
3,SpotifyCares,28280,48543,43265,3.246393
4,Delta,26166,45296,42253,3.345907


In [5]:
# 1. Get all conversation IDs involving SpotifyCares
spotify_conversation_ids = df[
    df["author_id"] == "SpotifyCares"
]["conversation_id"].dropna().unique()


# 2. Select 5 conversation IDs
sample_conversations = pd.Series(
    spotify_conversation_ids
).sample(30, random_state=42)


# 3. Display each conversation
for conversation_id in sample_conversations:

    print("=" * 100)
    print(f"Conversation: {conversation_id}")
    print("=" * 100)

    # Get all tweets belonging to this conversation
    conversation = df[
        df["conversation_id"] == conversation_id
    ].copy()

    # Sort chronologically
    conversation = conversation.sort_values("created_at")

    # Display required columns
    print(
        conversation[
            [
                "conversation_id",
                "tweet_id",
                "created_at",
                "author_id",
                "inbound",
                "text"
            ]
        ].to_string(index=False)
    )

    print("\n")

Conversation: 2583329
 conversation_id  tweet_id                     created_at    author_id  inbound                                                                                                                                                        text
         2583329   2583329 Thu Nov 16 23:18:30 +0000 2017       621440     True                                                                                                                  @115888 Im still waiting for reputation...
         2583329   2583327 Thu Nov 16 23:29:26 +0000 2017 SpotifyCares    False @621440 Hey there! Taylor Swift's 'Reputation' isn't available to stream just yet – stay tuned! For now, check out the singles: https://t.co/3h0TnPWP0c /GS
         2583329   2583328 Thu Nov 16 23:35:35 +0000 2017       621440     True                                                                                      @SpotifyCares Hi, see you in a few years on my Spotify internship.😘😘😘😘
         2583329   2583330 Thu Nov

In [6]:
# 1. Get conversation IDs involving SpotifyCares
spotify_conversation_ids = df[
    df["author_id"] == "SpotifyCares"
]["conversation_id"].dropna().unique()


# 2. Create spotify_df containing all tweets
#    belonging to those conversations
spotify_df = df[
    df["conversation_id"].isin(spotify_conversation_ids)
].copy()


# ============================================================
# 3. Inspect the dataset
# ============================================================

print("Shape:")
print(spotify_df.shape)


print("\nNumber of unique conversations:")
print(spotify_df["conversation_id"].nunique())


print("\nInbound value counts:")
print(spotify_df["inbound"].value_counts())


print("\nTop 10 authors:")
print(spotify_df["author_id"].value_counts().head(10))

Shape:
(91889, 8)

Number of unique conversations:
28280

Inbound value counts:
inbound
True     48543
False    43346
Name: count, dtype: int64

Top 10 authors:
author_id
SpotifyCares    43265
115888            332
hulu_support       49
125633             28
287348             24
215073             21
176622             21
AppleSupport       19
158590             19
220017             18
Name: count, dtype: int64
